# Sit & Goard (2023) on dandiset 000483: head rotation and remapping (Figures 2 and 3)

**Paper.** Sit KK, Goard MJ (2023), *Nature Communications* 14:1992,
[doi:10.1038/s41467-023-37704-5](https://doi.org/10.1038/s41467-023-37704-5); claims file
written from the version of record without its supplement.

**Dataset.** [DANDI:000483](https://dandiarchive.org/dandiset/000483/0.230421.2321), version
0.230421.2321, the only published version (the paper cites none). See `single_cue.ipynb` for the
data check of all files.

| Claim | Here |
|---|---|
| fig2f-head-chamber-same-preference | tested |
| fig3c-coherent-remapping | tested |

Activity is inferred spikes computed from the raw fluorescence (no neuropil correction; see
`rsc.py`). Cells are selected by the heading-selection test with the centred fit, the reading
adopted in `single_cue.ipynb` (the code's uncentred fit only passes cells whose peak lies near
one heading, so preferred headings cannot be compared across conditions with it).

In [ ]:
import time
import warnings
from multiprocessing import Pool

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import rsc

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
started = time.time()
cat = rsc.catalog(full=False)
sub = cat[cat.experiment.isin(["head_rotation", "chamber_rotation", "remapping"])].reset_index(drop=True)
recs = rsc.load_many(sub.url.tolist())
with Pool(4) as pool:
    for r in recs:
        r["dff"] = rsc.dff(r["F"])
        r["sp"] = rsc.spikes(r["dff"], pool)
        del r["F"]
print(f"dandiset {rsc.DANDISET_ID} version {rsc.DANDISET_VERSION}: {len(sub)} files loaded and deconvolved "
      f"in {time.time() - started:.0f} s")

In [ ]:
rows = []
for m, r in zip(sub.itertuples(), recs):
    h = rsc.clean_heading(r["heading"])
    step = rsc.wrap180(np.diff(h))
    rows.append(dict(experiment=m.experiment, subject=m.subject, identifier=m.identifier[:6] + " " + m.identifier.split("_")[-1],
                     frames=m.n_frames, rois=m.n_rois, heading_nan=int(np.isnan(r["heading"]).sum()),
                     heading_range=f"{np.nanmin(r['heading']):.0f}..{np.nanmax(r['heading']):.0f}",
                     median_step=round(float(np.median(step)), 2), direction=int(np.sign(np.median(step))),
                     trials=len(rsc.trial_segments(r["heading"])),
                     spikes_per_s=round(float(r["sp"].sum(0).mean() / (m.n_frames / 10)), 3)))
info = pd.DataFrame(rows)
print(info.to_string(index=False))

Head-rotation (HR) and chamber-rotation (CR) files come in pairs with the same number of ROIs (a
single set of ROIs, Results), as do the two contexts of each remapping pair. In the head-rotation files the
heading steps 0.7 to 1.3 degrees per frame (7 to 13 degrees/s; the Methods give 2 rpm, 12 degrees/s). In 5 of the 7 HR/CR pairs the
stored heading runs in opposite directions in the two files; in two pairs (210427 SKKS091
and 210429 SKKS089) it runs the same way. The paper reports 5 pairs
over 4 mice; the 5 opposite-direction pairs are 5 over 4 mice.

**Clock check.** These files rotate one way at constant speed, where a time offset between heading
and activity only rotates every tuning curve and cannot be seen within a file; the check of
`single_cue.ipynb` on the files that rotate both ways is the only one, and the clock of these
files is not established. An offset common to the two files of a pair would cancel in the
differences tested below; an offset that differs between them would not.

In [ ]:
trials = [rsc.trial_tuning(r["sp"], r["heading"]) for r in recs]
t0 = time.time()
with Pool(4) as pool:
    sel = rsc.selection_many(trials, "single", 201, pool, centre=True)
info["selective"] = [s.is_hd.sum() for s in sel]
info["trials_used"] = [t.shape[2] for t in trials]
print(f"selection test in {time.time() - t0:.0f} s")
print(info[["experiment", "subject", "identifier", "rois", "trials_used", "selective"]].to_string(index=False))

## fig2f-head-chamber-same-preference

> Matched ROIs have on average the same preferred heading relative to the wall pattern whether
> the head is rotated in a fixed chamber or the chamber is rotated around a fixed head (V-test
> against 0 degrees, V = 14.1, p = 2.5e-4), though not all cells keep their preference.

Per ROI, preferred heading in HR minus that in CR, wrapped to +-180, V-test against 0. Readings of
the claims file: cells (a) selective in both, (b) in either, (c) in CR; preferred heading (i)
peak of the trial-mean curve, (ii) the code's 'fit' method. Two further details are open and
both are run: the pairs, all 7 or the 5 whose heading runs in opposite directions; and the sign
of the heading. The paper says that in both conditions "the visual cue card [moves] across the
visual field in the same direction" (Figure 2B legend), while the stored headings of the 5 pairs
run in opposite directions, so one of the two must be mirrored to put both in the frame of the
wall pattern; the reading "CR mirrored" (heading -h in the CR file) does this, "as stored" does
not.

In [ ]:
pairs = []
for i in sub.index[sub.experiment == "head_rotation"]:
    n = int(sub.identifier[i][-3:])
    j = sub.index[(sub.experiment == "chamber_rotation") & (sub.date == sub.date[i]) & (sub.subject == sub.subject[i])
                  & (sub.identifier.str[-3:].astype(int) == n + 1)][0]
    pairs.append((i, j, info.direction[i] != info.direction[j]))
print("pairs (HR, CR, opposite directions):", [(info.identifier[i], info.identifier[j], o) for i, j, o in pairs])

def preference(i, method, mirror=False):
    tc = trials[i].mean(2)
    if mirror:
        tc = tc[:, ::-1]
    return rsc.peak_heading(tc) if method == "peak" else rsc.fit_preferred(tc, "single")

res = []
diffs = {}
for method in ["peak", "fit"]:
    for mirror in [True, False]:
        for cells in ["both", "either", "CR"]:
            for pset in ["5 pairs", "7 pairs"]:
                d_all = []
                for i, j, opp in pairs:
                    if pset == "5 pairs" and not opp:
                        continue
                    h1, h2 = sel[i].is_hd.values, sel[j].is_hd.values
                    m = {"both": h1 & h2, "either": h1 | h2, "CR": h2}[cells]
                    d_all.append(rsc.wrap180(preference(i, method) - preference(j, method, mirror))[m])
                d = np.concatenate(d_all)
                V, p, n = rsc.vtest(d)
                diffs[(method, mirror, cells, pset)] = d
                res.append(dict(preference=method, sign="CR mirrored" if mirror else "as stored", cells=cells, pairs=pset,
                                n=n, V=round(V, 1), p=float(f"{p:.3g}"),
                                within_30deg=round(float(np.mean(np.abs(d) <= 30)), 3)))
res = pd.DataFrame(res)
print(res.to_string(index=False))
print("\nReadings with p < 0.05 and V > 0:", int(((res.p < 0.05) & (res.V > 0)).sum()), "of", len(res))
print("Closest n to the paper's 32:", res.iloc[(res.n - 32).abs().argsort()[:3]][["preference", "sign", "cells", "pairs", "n", "V", "p"]].to_string(index=False))

In [ ]:
print("Per pair, peak preference, CR mirrored, cells selective in both:")
for i, j, opp in pairs:
    m = sel[i].is_hd.values & sel[j].is_hd.values
    d = rsc.wrap180(preference(i, "peak") - preference(j, "peak", True))[m]
    V, p, n = rsc.vtest(d) if len(d) else (np.nan, np.nan, 0)
    print(f"  {info.identifier[i]} / {info.identifier[j]} opposite={opp}: n={n}, V={V:.1f}, p={p:.2g}, "
          f"median |diff|={np.median(np.abs(d)) if len(d) else np.nan:.0f} deg")
# Null for the V statistic: ROIs paired at random within each pair of files (the paper's shuffle)
rng = np.random.default_rng(202)
null = []
for _ in range(200):
    d_all = []
    for i, j, opp in pairs:
        if not opp:
            continue
        m = np.flatnonzero(sel[i].is_hd.values & sel[j].is_hd.values)
        d_all.append(rsc.wrap180(preference(i, "peak")[m] - preference(j, "peak", True)[rng.permutation(m)]))
    null.append(rsc.vtest(np.concatenate(d_all))[0])
print(f"Random pairing of ROIs (5 pairs, both, peak, CR mirrored): V mean {np.mean(null):.1f}, 95th percentile {np.percentile(null, 95):.1f}")
d = diffs[("peak", True, "both", "5 pairs")]
fig, ax = plt.subplots(figsize=(4, 3))
ax.hist(d, bins=36, range=(-180, 180), color="0.4"); ax.set_xlabel("HR minus CR preferred heading (deg)"); ax.set_ylabel("ROIs")
plt.tight_layout(); plt.show()
print("All differences lie in -180 to 180.")

Further checks asked for in verification: the random-pairing null per pair (1000 draws), the
activity and the share of selective cells in the two files of each pair (heading-locked
artefacts of derotation would make the head-rotation files look more selective), and how far a
clock offset could move the preferred headings. In `single_cue.ipynb` the lag of the peak
correlation is 4 to 6 frames in the files that rotate both ways, with one file at -6; an offset
of k frames moves every preferred heading of a file by k times the heading step per frame.

In [ ]:
pref_hr = {i: preference(i, "peak") for i, _, _ in pairs}
pref_cr = {j: preference(j, "peak", True) for _, j, _ in pairs}
rng = np.random.default_rng(204)
rows2 = []
for i, j, opp in pairs:
    m = np.flatnonzero(sel[i].is_hd.values & sel[j].is_hd.values)
    if len(m) == 0:
        continue
    V = rsc.vtest(rsc.wrap180(pref_hr[i][m] - pref_cr[j][m]))[0]
    nullp = np.array([rsc.vtest(rsc.wrap180(pref_hr[i][m] - pref_cr[j][rng.permutation(m)]))[0] for _ in range(1000)])
    rows2.append(dict(pair=f"{info.identifier[i]} / {info.identifier[j].split(' ')[-1]}", opposite=opp, n=len(m), V=round(V, 1),
                      null_mean=round(float(nullp.mean()), 1), null_p95=round(float(np.percentile(nullp, 95)), 1),
                      p_random=round(float(np.mean(nullp >= V)), 3),
                      HR_selective=f"{sel[i].is_hd.sum()}/{len(sel[i])} ({100 * sel[i].is_hd.mean():.0f}%)",
                      CR_selective=f"{sel[j].is_hd.sum()}/{len(sel[j])} ({100 * sel[j].is_hd.mean():.0f}%)",
                      HR_spikes_s=info.spikes_per_s[i], CR_spikes_s=info.spikes_per_s[j],
                      HR_step=abs(info.median_step[i]), CR_step=abs(info.median_step[j]),
                      shift_6_frames_deg=f"{6 * abs(info.median_step[i]):.0f} / {6 * abs(info.median_step[j]):.0f}"))
print(pd.DataFrame(rows2).to_string(index=False))
print("p_random: share of 1000 random pairings of the same ROIs with V at least the real one (peak preference, CR mirrored, cells selective in both)")

**Result: partially reproduced, resting on assumptions (the CR heading mirrored, the 5 opposite-direction pairs, the centred fit).** With the chamber-rotation heading mirrored, cells selective in both conditions in the 5 pairs give V = 34.6, n = 228, p = 5.9e-04 (peak; 'fit' V = 36.0, p = 3.8e-04), so the criterion's test passes; cells selective in CR give V = 47.7, p = 7.3e-05. With all 7 pairs no reading has p < 0.05, and with the headings as stored every V is negative: 4 of 24 readings pass. The V is not the paper's 14.1 (n = 228 against 32). Matched ROIs do not keep their preference better than ROIs paired at random: over the 5 pairs random pairing gives a larger V (mean 46.2, 95th percentile 57.6), and in every pair the real V lies within its own random-pairing null (share of random pairings at or above it 0.21 to 0.998; SKKS116, V = 58.9 against a null mean of 56.8). The V-test passes because preferred headings are not uniform in either file. Head-rotation files have more selective cells than their chamber-rotation partners (53 to 83% against 8 to 65%) and more activity (0.095 to 3.19 spikes/s against 0.063 to 0.114), so heading-locked artefacts of derotation are not excluded. A clock offset of 6 frames would move preferred headings by 4 to 8 degrees in a head-rotation file and 12 degrees in a chamber-rotation file.

## fig3c-coherent-remapping

> When all the wall cues are replaced, heading cells remap coherently, each shifting its
> preferred heading by about the same angle as the rest of the population in its recording
> (V-test of the Delta phase offsets against 0 degrees, V = 25.6, p = 4.9e-6).

Pairs: the two remapping files of a mouse on a day (same ROIs), in the order recorded. Per
neuron, Delta phase offset = (pref A - pref B) minus the expected offset of the other neurons:
(i) the median of their differences, (ii) the median of their pref A minus the median of their
pref B. Medians of angles are circular medians (the angle minimising the summed absolute
circular distance, a standard choice). Cells (a) selective in both contexts, (b) in either;
preference, peak or 'fit'. Null: neurons paired at random within each recording (Methods).

In [ ]:
rm = sub[sub.experiment == "remapping"]
rpairs = [tuple(g.sort_values("identifier").index) for _, g in rm.groupby(["date", "subject"])]
print("remapping pairs (subject, file A, file B):", [(sub.subject[i], info.identifier[i], info.identifier[j]) for i, j in rpairs])
grid = np.arange(-180, 180, 1.0)

def circmed(x):
    return grid[np.argmin(np.abs(rsc.wrap180(np.asarray(x)[None] - grid[:, None])).sum(1))]

def delta_offsets(P, cells, expected, rng=None):
    out, rec = [], []
    for k, (i, j) in enumerate(rpairs):
        m = {"both": sel[i].is_hd.values & sel[j].is_hd.values, "either": sel[i].is_hd.values | sel[j].is_hd.values}[cells]
        idx = np.flatnonzero(m)
        if len(idx) < 3:
            continue
        pa, pb = P[i][idx], P[j][idx]
        if rng is not None:
            pb = pb[rng.permutation(len(pb))]
        diff = rsc.wrap180(pa - pb)
        for n in range(len(idx)):
            o = np.delete(np.arange(len(idx)), n)
            e = circmed(diff[o]) if expected == "i" else rsc.wrap180(circmed(pa[o]) - circmed(pb[o]))
            out.append(rsc.wrap180(diff[n] - e)); rec.append(k)
    return np.array(out), np.array(rec)

rows = []
for method in ["peak", "fit"]:
    P = {i: (rsc.peak_heading(trials[i].mean(2)) if method == "peak" else rsc.fit_preferred(trials[i].mean(2))) for p in rpairs for i in p}
    for cells in ["both", "either"]:
        for expected in ["i", "ii"]:
            d, rec = delta_offsets(P, cells, expected)
            V, p, n = rsc.vtest(d)
            rows.append(dict(preference=method, cells=cells, expected=expected, n=n, recordings=len(np.unique(rec)),
                             V=round(V, 1), p=float(f"{p:.3g}"), within_30deg=round(float(np.mean(np.abs(d) <= 30)), 3)))
            if (method, cells, expected) == ("peak", "both", "i"):
                d_main, rec_main, P_main = d, rec, P
rows = pd.DataFrame(rows)
print(rows.to_string(index=False))
print("\nReadings with p < 0.05 and V > 0:", int(((rows.p < 0.05) & (rows.V > 0)).sum()), "of", len(rows))
rng = np.random.default_rng(203)
null = [rsc.vtest(delta_offsets(P_main, "both", "i", rng)[0])[0] for _ in range(1000)]
print(f"Random pairing of neurons (peak, both, (i)), 1000 draws: V mean {np.mean(null):.1f}, 95th percentile {np.percentile(null, 95):.1f}, "
      f"share at or above the real V {np.mean(np.array(null) >= rows.V[0]):.3f}")
print("Per recording pair (peak, both, (i)):")
for k, (i, j) in enumerate(rpairs):
    dk = d_main[rec_main == k]
    if len(dk):
        print(f"  {sub.subject[i]} {info.identifier[i]}: n={len(dk)}, V={rsc.vtest(dk)[0]:.1f}, median |offset|={np.median(np.abs(dk)):.0f} deg")
    else:
        print(f"  {sub.subject[i]} {info.identifier[i]}: fewer than 3 cells selective in both")
fig, ax = plt.subplots(figsize=(4, 3))
ax.hist(d_main, bins=36, range=(-180, 180), color="0.4"); ax.set_xlabel("Delta phase offset (deg)"); ax.set_ylabel("neurons")
plt.tight_layout(); plt.show()
print("All offsets lie in -180 to 180.")

**Result: partially reproduced, under the centred fit (an assumption).** All 8 readings give V > 0 with p < 0.05. For cells selective in both contexts, peak preference and the median of the other cells' differences: V = 47.0, n = 85 over 9 recording pairs, p = 2.8e-13, against 25.6 with n = 66 in the paper, outside the tolerance of 2.56; no reading gives a V within it (38.4 to 233.5). Pairing neurons at random gives V of 36.2 on average (95th percentile 45.7, 1000 draws); 2.6% of random pairings reach the real V, so the real V lies only just above the null. 8 of the 9 pairs with 3 or more cells have V > 0.

## Summary

| Claim | Outcome |
|---|---|
| fig2f-head-chamber-same-preference | partially reproduced, assumed (CR heading mirrored, 5 pairs, centred fit): the V-test passes, but random pairing gives as large a V in every pair |
| fig3c-coherent-remapping | partially reproduced, assumed (centred fit): every reading passes; V outside the tolerance and just above the random-pairing null |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's: inferred spikes from raw fluorescence without neuropil correction; cells selected with the centred fit. For Figure 2, the sign of the chamber-rotation heading and the set of pairs are not stated and were chosen as said above. Circular medians stand in for the medians of the Methods.

What was found about the dataset: 7 head-rotation pairs over 5 mice where the paper reports 5 over 4. The stored headings of 5 pairs run in opposite directions in the two conditions, and the files do not say how either heading is defined. Remapping files are named "1" and "2" or "Stars" and "Stripes"; one pair of SKKS091 is named "1-001" and "1-002", so it may not be a remapping pair; it enters only the readings with cells selective in either context, as it has fewer than 3 cells selective in both.